## Détection d'anomalies sur l'indicateur de Profitabilité

### Notebook 01 — Intégration des données

---

### Objectif

Ce notebook constitue la première étape du pipeline de données.

L’objectif est de regrouper les différents fichiers CSV disponibles dans un
dataset analytique unique, tout en conservant l’information de provenance
de chaque observation.

---

In [2]:
# ============================================================
# Import des bibliothèques
# ============================================================
#

from pathlib import Path
import pandas as pd

In [10]:
# ============================================================
# Définition des répertoires
# ============================================================

PROJECT_DIR = Path.cwd().parent

RAW_DATA = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA = PROJECT_DIR / "data" / "processed"

PROCESSED_DATA.mkdir(
    parents=True,
    exist_ok=True
)



In [11]:
# Identification des fichiers sources

csv_files = sorted(
    RAW_DATA.glob("*.csv")
)

if not csv_files:
    raise FileNotFoundError(
        f"Aucun fichier CSV trouvé dans : {RAW_DATA}"
    )

print("=" * 70)
print("FICHIERS SOURCES DETECTES")
print("=" * 70)

print(f"Nombre de fichiers : {len(csv_files)}")

for file in csv_files:
    print(f"• {file.name}")

FICHIERS SOURCES DETECTES
Nombre de fichiers : 6
• electric_bulbs_2025_LMFR.csv
• electric_bulbs_2026_LMFR.csv
• textile_2025_LMFR.csv
• textile_2026_LMFR.csv
• white_paints_2025_LMFR.csv
• white_paints_2026_LMFR.csv


In [12]:
# ============================================================
# Lecture des données sources
# ============================================================

def load_dataset(filepath: Path) -> pd.DataFrame:

    df = pd.read_csv(
        filepath,
        sep=",",
        encoding="utf-8"
    )

    df["Source_File"] = filepath.stem

    return df

In [13]:
# ============================================================
# Chargement des jeux de données
# ============================================================

datasets = {}

for file in csv_files:

    datasets[file.stem] = load_dataset(file)

print("Chargement terminé.\n")
for name, data in datasets.items():

    print(
        f"{name:<35} "
        f"{len(data):>8,} lignes | "
        f"{len(data.columns):>3} colonnes"
    )

Chargement terminé.

electric_bulbs_2025_LMFR              13,049 lignes |  15 colonnes
electric_bulbs_2026_LMFR               7,663 lignes |  15 colonnes
textile_2025_LMFR                     29,630 lignes |  15 colonnes
textile_2026_LMFR                     20,025 lignes |  15 colonnes
white_paints_2025_LMFR                 2,722 lignes |  15 colonnes
white_paints_2026_LMFR                 1,799 lignes |  15 colonnes


# Fusion des jeux de données

Les six jeux de données sont regroupés verticalement afin de constituer
un dataset analytique unique.


In [15]:
# ============================================================
# Fusion des jeux de données
# ============================================================

dataset_raw = pd.concat(
    datasets.values(),
    ignore_index=True,
    sort=False
)

print("=" * 70)
print("DATASET ANALYTIQUE INTEGRE")
print("=" * 70)

print(f"Lignes    : {len(dataset_raw):,}")
print(f"Colonnes  : {dataset_raw.shape[1]:,}")

DATASET ANALYTIQUE INTEGRE
Lignes    : 74,888
Colonnes  : 15


In [16]:
# ============================================================
# Contrôle de provenance
# ============================================================

source_summary = (
    dataset_raw
    .groupby("Source_File")
    .size()
    .reset_index(name="Nombre_Lignes")
    .sort_values("Source_File")
)

display(source_summary)

,Source_File,Nombre_Lignes
0,electric_bulbs_2025_LMFR,13049
1,electric_bulbs_2026_LMFR,7663
2,textile_2025_LMFR,29630
3,textile_2026_LMFR,20025
4,white_paints_2025_LMFR,2722
5,white_paints_2026_LMFR,1799


# Sauvegarde

Le dataset fusionné est enregistré dans le dossier `processed`.


In [19]:
# ============================================================
# Export du dataset analytique
# ============================================================

output_path = PROCESSED_DATA / "dataset_raw.csv"

dataset_raw.to_csv(
    output_path,
    index=False,
    encoding="utf-8-sig"
)
print(f"Dataset enregistré")
print(f"Dimensions : {dataset_raw.shape[0]:,} lignes × {dataset_raw.shape[1]} colonnes")

Dataset enregistré
Dimensions : 74,888 lignes × 15 colonnes


# Conclusion

À l'issue de cette première étape :

- les six jeux de données ont été importés ;
- leur structure a été harmonisée ;
- les données ont été regroupées dans un dataset unique ;
- le dataset intégré contient 74 888 observations et 15 colonnes ;
- aucune transformation métier n'a encore été réalisée.

Le dataset `dataset_raw.csv` constitue la base utilisée par le notebook
suivant, consacré au contrôle de la qualité des données.